In [1]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from datasets import load_dataset

ds = load_dataset("PotatoHD/ru-text-corpus")['train'].to_pandas()
ds = ds[ds.src == 'news'].copy().sample(frac=0.05) #корпус очень большой, возьмем 5% записей это примерно 1 млн. слов

In [12]:
import re
import pandas as pd

def to_lower(text):
    return str(text).lower() if pd.notna(text) else text


def remove_punct(text):
    return re.sub(r"[^\w\s]", " ", str(text), flags=re.UNICODE) if pd.notna(text) else text


def to_tokens(text):
    return re.findall(r"\w+", str(text), flags=re.UNICODE) if pd.notna(text) else []

def tokenize(text):
    return to_tokens(remove_punct(to_lower(text)))

In [13]:
ds['tokens'] = ds.text.apply(to_lower).apply(remove_punct).apply(to_tokens)
ds.head(5)

,text,src,tokens
55117,Британское издание Daily Express (DE) сообщает...,news,"[британское, издание, daily, express, de, сооб..."
34206,Второй по величине банк Германии Commerzbank A...,news,"[второй, по, величине, банк, германии, commerz..."
50363,Не стихают обсуждения боя между одним из лучши...,news,"[не, стихают, обсуждения, боя, между, одним, и..."
32203,"Израильский город Шаар-ха-Негев, находящийся в...",news,"[израильский, город, шаар, ха, негев, находящи..."
54261,Разведывательная служба Израиля распространила...,news,"[разведывательная, служба, израиля, распростра..."


In [14]:
#Статистика: число слов, размер словаря, двадцать самых частых слов.
print("Число слов", ds.tokens.str.len().sum())
print("Размер словаря", len(ds.tokens.explode().unique()))
print("Двадцать самых частых слов.", ds.tokens.explode().value_counts().head(20))

Число слов 1345153
Размер словаря 123282
Двадцать самых частых слов. tokens
в      57573
и      33938
на     25414
не     17325
что    15416
с      15247
по     12500
а       7119
из      6852
за      6618
к       5955
как     5555
но      5505
о       5328
это     4928
он      4849
для     4723
от      4334
его     4311
у       4044
Name: count, dtype: int64


In [15]:
ds_train = ds.sample(frac=0.9)
ds_test = ds.drop(ds_train.index)

del ds #освобождаю ресурсы для экономии оперативки

In [16]:
import random 
import numpy as np 

class CounterLM:
    
    def __init__(self, n=2):
        self.prob_table = None
        self.words_freq = None
        self.n = n
        self.V = 0
        

    @staticmethod
    def to_ngram(tokens, n=2):
        tokens_len = len(tokens)
        return [tuple(tokens[i:i+n]) for i in range( tokens_len-n+1)]


    def fit(self, text_df, tokens):
        prob_table = text_df[tokens].apply(self.to_ngram).explode().value_counts().rename("c_vw").to_frame()
        self.words_freq = text_df[tokens].explode().value_counts()
        self.V = text_df[tokens].explode().nunique()

        prob_table[["v", "w"]] = pd.DataFrame(prob_table.index.tolist(), index=prob_table.index)
        prob_table["c_v"]  = prob_table["v"].map(self.words_freq)
        prob_table["prob"] = prob_table["c_vw"] / prob_table["c_v"]

        self.prob_table = prob_table.reset_index()

        

    def prob(self, v, w):
        prob = self.prob_table[(self.prob_table.v == v) & (self.prob_table.w == w)].prob
        if prob.shape[0] > 0:
            return prob.iloc[0]
        else:
            return 0.0

    def generate(self, start, output_len=30, T=1):
        output_text = [start]
        for i in range(output_len):
            #print(output_text)
            probs = self.prob_table[self.prob_table.v == output_text[-1]]
            tokens = probs.w.tolist()
            wth = probs.prob.pow(1.0/T).tolist()

            output_text += random.choices(tokens, wth)
        return ' '.join(output_text)

    def logprob(self, text, alpha=1):
        logP = 0.0

        for i in range(len(text) - self.n + 1):
            v,w = text[i:i+self.n]

            c_vw = self.prob_table[(self.prob_table.v == v) & (self.prob_table.w == w)].c_vw
            if c_vw.shape[0] > 0:
                c_vw = c_vw.iloc[0]
            else:
                c_vw = 0.0
            c_v = self.words_freq['v'] 

            logP += np.log((c_vw+alpha) / (c_v + alpha * self.V))

        return logP
        


In [17]:
"""
Сглаживание лапласа помогает избавится от случаев с -inf оценкой
 правдоподобия для биграм, которых не было в датасете. Если 
попадается хотя-бы одна такая биграма то вся оценка правдоподобия будет -inf, 
а две оценки с -inf не сравнимы. Сглаживание помогает "прикинуть" 
редкость для пары который нет на основе размера словаря. 
"""


'\nСглаживание лапласа помогает избавится от случаев с -inf оценкой\n правдоподобия для биграм, которых не было в датасете. Если \nпопадается хотя-бы одна такая биграма то вся оценка правдоподобия будет -inf, \nа две оценки с -inf не сравнимы. Сглаживание помогает "прикинуть" \nредкость для пары который нет на основе размера словаря. \n'

In [18]:
lm = CounterLM(n=2)

lm.fit(ds_train, tokens='tokens')



In [19]:
for i in range(5):
    print(lm.generate(start='анекдот', T=1.5))

анекдот если смотреть на глазах у вас уверяю вас как говорится любовь бизнесмена были конкурентными в прежних уровнях изменилась в криминальных городов авторам этой системой компания приобрела как то время воспользовавшись
анекдот если дельнери узнал о собственных шаблонов и белорусских тюрьмах и партнеры шли к левому фронту открыто поддерживал с удовольствием оправился ему препятствия это гособвинитель следует пересмотреть рейтинги держатся мякяряйнен лучшей
анекдот то ли стилистика такая разная сложнее так его зарплата снизились еще на сохранность их защиту титула но унижать политзаключенного зборовский угрожал применением насилия продолжал боксировать много проблем я другая структура
анекдот в плохом отношении аэрофлота дочерних изданий в италию команды георгию кондратьеву пришлось стать третьим по габаритам с театром одеон и поднималась в хуле погибли после атаки после отправить за отопления
анекдот о темах своего дмитрий сухоплюев пригрозил им удалось пронести на острове джерба не

In [20]:
for i in range(5):
    print(lm.generate(start='анекдот', T=0.5))

анекдот если бы не только в этом году когда он в котором подозревают в начале июня в том что в этом в сша и в то что российские болельщики приходят в
анекдот если бы не был в целом по его словам когда он не только в том что в частности в этом году в понедельник в четверг в ходе судебного следствия в
анекдот если же время как и не в том числе и в этом году в этом матче с динамо москва в сборной россии и не менее чем на фоне обсуждения того
анекдот в ожидании гимна но уже не будет ли он в этом году в конце августа на территории россии и баста и в этом матче с учетом того как и не
анекдот о россии и не более того что россия и все все таки не было принято решение о том что в том что в пятницу в его словам представителя пресс конференции


In [21]:
for i in range(5):
    print(lm.generate(start='анекдот', T=0.1))

анекдот в том что в том что в том что в том что в том что в том что в этом году в том что в том что в том что
анекдот в том что в том что в том что в том что в том что в том что в том что в этом году в том что в том что
анекдот в россии в том что в том что в том что в том что в том что в том что в том что в том что в том что в
анекдот в том что в том что в этом году в том что в том что в том что в том что в том что в том что в том что
анекдот в том что в том что в том что в том что в том что в том что в том что в том что в том что в том что


In [22]:
for text in ds_test.sample(5).text:
    
    text_tokens = tokenize(text)
    
    text_tokens_shuffle = text_tokens.copy()
    random.shuffle(text_tokens_shuffle)

    print(lm.logprob(text_tokens, alpha=0.001), 'vs' ,lm.logprob(text_tokens_shuffle, alpha=0.001))

-4391.180261952396 vs -6104.263653889404
-5324.8718793996395 vs -6361.352043052311
-3731.0099550282334 vs -5138.350132653657
-5571.391764479487 vs -7036.358392151091
-5093.391572861102 vs -6198.114645899234


In [24]:
from tqdm import tqdm
for alpha in [1,0.1,0.01]:
    perplexity = 0
    tokens = 0

    for text in tqdm(ds_test.text):
        text_tokens = tokenize(text)

        perplexity += lm.logprob(text_tokens, alpha=alpha)
        tokens += len(text_tokens)

    print( "alpha =", alpha, ':', 'perplexity', perplexity/tokens) 

100%|██████████| 214/214 [13:45<00:00,  3.86s/it]


alpha = 1 : perplexity -10.485102359825321


100%|██████████| 214/214 [15:18<00:00,  4.29s/it]


alpha = 0.1 : perplexity -9.511758161875575


100%|██████████| 214/214 [15:17<00:00,  4.29s/it]

alpha = 0.01 : perplexity -8.45690880377456
